In [1]:
import pandas as pd
from pathlib import Path


In [2]:

# set the folder path 
data_folder = Path.home() / "Downloads" / "esg data "

#load file 
sites = pd.read_csv(data_folder / "site_master.csv")
emissions = pd.read_csv(data_folder / "emissions_data.csv")
hr = pd.read_csv(data_folder / "hr_social_data.csv")

#quick check
print("sites rows:", sites.shape)
print("emissions rows:", emissions.shape)
print("hr rows:", hr.shape)

sites rows: (10, 9)
emissions rows: (20, 8)
hr rows: (20, 11)


In [3]:
sites.head()

,site_id,company_id,site_name,country,state,city,site_type,business_unit,employees_site
0,S001,1,Houston Plant,USA,Texas,Houston,Manufacturing,Operations,1850
1,S002,1,Chicago Office,USA,Illinois,Chicago,Office,Corporate,650
2,S003,1,Ohio Distribution Center,USA,Ohio,Columbus,Warehouse,Logistics,920
3,S004,1,Atlanta Plant,USA,Georgia,Atlanta,Manufacturing,Operations,1600
4,S005,1,Phoenix Office,USA,Arizona,Phoenix,Office,Sales,430


In [4]:
emissions.head()

,record_id,site_id,reporting_year,month,scope_1_tco2e,scope_2_tco2e,scope_3_tco2e,total_tco2e
0,1,S001,2024,1,410,295,870,1575
1,2,S001,2024,2,398,287,845,1530
2,3,S004,2024,1,365,250,720,1335
3,4,S004,2024,2,372,258,735,1363
4,5,S006,2024,1,345,235,690,1280


In [5]:
hr.head()

,record_id,site_id,reporting_year,month,total_employees,female_employees,female_management_pct,training_hours,total_turnover_pct,lost_time_injury_rate,recordable_incidents
0,1,S001,2024,1,1850,520,23,760,3.8,2.1,5
1,2,S001,2024,2,1848,522,23,780,3.6,1.9,4
2,3,S004,2024,1,1600,470,24,690,3.4,1.8,4
3,4,S004,2024,2,1602,472,24,705,3.5,1.7,3
4,5,S006,2024,1,1450,390,22,620,3.9,2.3,5


In [8]:
# group the emissions table by site_id
# this means we want one final row for each site

site_emissions = emissions.groupby("site_id", as_index=False).agg({
    "scope_1_tco2e": "sum",   # add all monthly scope 1 values for each site
    "scope_2_tco2e": "sum",   # add all monthly scope 2 values for each site
    "scope_3_tco2e": "sum",   # add all monthly scope 3 values for each site
    "total_tco2e": "sum"      # add all monthly total emissions for each site
})

# show the first few rows to check the result
site_emissions.head()

,site_id,scope_1_tco2e,scope_2_tco2e,scope_3_tco2e,total_tco2e
0,S001,808,582,1715,3105
1,S002,34,102,129,265
2,S003,142,172,503,817
3,S004,737,508,1455,2698
4,S005,19,62,78,159


In [9]:
# group the HR table by site_id
# again, the goal is one final row for each site

site_hr = hr.groupby("site_id", as_index=False).agg({
    "total_employees": "mean",           # average employee count across months
    "female_employees": "mean",          # average female employee count
    "female_management_pct": "mean",     # average female management percentage
    "training_hours": "sum",             # total training hours across months
    "total_turnover_pct": "mean",        # average turnover rate
    "lost_time_injury_rate": "mean",     # average injury rate
    "recordable_incidents": "sum"        # total incidents across months
})

# check the result
site_hr.head()

,site_id,total_employees,female_employees,female_management_pct,training_hours,total_turnover_pct,lost_time_injury_rate,recordable_incidents
0,S001,1849.0,521.0,23.0,1540,3.70,2.00,9
1,S002,651.0,311.0,38.0,566,1.75,0.20,0
2,S003,919.0,295.5,31.0,732,2.75,1.05,4
3,S004,1601.0,471.0,24.0,1395,3.45,1.75,7
4,S005,431.0,190.5,35.0,346,1.95,0.10,0


In [10]:
# start with the site master table because it contains site names and location info
site_summary = sites.merge(site_emissions, on="site_id", how="left")

# then add the HR summary into the same table
site_summary = site_summary.merge(site_hr, on="site_id", how="left")

# check the final merged site table
site_summary.head()

,site_id,company_id,site_name,country,state,city,site_type,business_unit,employees_site,scope_1_tco2e,scope_2_tco2e,scope_3_tco2e,total_tco2e,total_employees,female_employees,female_management_pct,training_hours,total_turnover_pct,lost_time_injury_rate,recordable_incidents
0,S001,1,Houston Plant,USA,Texas,Houston,Manufacturing,Operations,1850,808,582,1715,3105,1849.0,521.0,23.0,1540,3.70,2.00,9
1,S002,1,Chicago Office,USA,Illinois,Chicago,Office,Corporate,650,34,102,129,265,651.0,311.0,38.0,566,1.75,0.20,0
2,S003,1,Ohio Distribution Center,USA,Ohio,Columbus,Warehouse,Logistics,920,142,172,503,817,919.0,295.5,31.0,732,2.75,1.05,4
3,S004,1,Atlanta Plant,USA,Georgia,Atlanta,Manufacturing,Operations,1600,737,508,1455,2698,1601.0,471.0,24.0,1395,3.45,1.75,7
4,S005,1,Phoenix Office,USA,Arizona,Phoenix,Office,Sales,430,19,62,78,159,431.0,190.5,35.0,346,1.95,0.10,0


In [11]:
# create female workforce percentage
# formula: female employees / total employees
site_summary["female_workforce_pct"] = (
    site_summary["female_employees"] / site_summary["total_employees"]
) * 100

# create training hours per employee
# formula: total training hours / total employees
site_summary["training_hours_per_employee"] = (
    site_summary["training_hours"] / site_summary["total_employees"]
)

# round values to 2 decimal places for easier reading
site_summary = site_summary.round(2)

# check result
site_summary.head()

,site_id,company_id,site_name,country,state,city,site_type,business_unit,employees_site,scope_1_tco2e,...,total_tco2e,total_employees,female_employees,female_management_pct,training_hours,total_turnover_pct,lost_time_injury_rate,recordable_incidents,female_workforce_pct,training_hours_per_employee
0,S001,1,Houston Plant,USA,Texas,Houston,Manufacturing,Operations,1850,808,...,3105,1849.0,521.0,23.0,1540,3.70,2.00,9,28.18,0.83
1,S002,1,Chicago Office,USA,Illinois,Chicago,Office,Corporate,650,34,...,265,651.0,311.0,38.0,566,1.75,0.20,0,47.77,0.87
2,S003,1,Ohio Distribution Center,USA,Ohio,Columbus,Warehouse,Logistics,920,142,...,817,919.0,295.5,31.0,732,2.75,1.05,4,32.15,0.80
3,S004,1,Atlanta Plant,USA,Georgia,Atlanta,Manufacturing,Operations,1600,737,...,2698,1601.0,471.0,24.0,1395,3.45,1.75,7,29.42,0.87
4,S005,1,Phoenix Office,USA,Arizona,Phoenix,Office,Sales,430,19,...,159,431.0,190.5,35.0,346,1.95,0.10,0,44.20,0.80


In [12]:
# create an output folder if it does not already exist
output_folder = data_folder / "output"
output_folder.mkdir(exist_ok=True)

# save the final clean table as a csv file
site_summary.to_csv(output_folder / "clean_site_summary.csv", index=False)

# confirmation message
print("clean_site_summary.csv saved successfully")

clean_site_summary.csv saved successfully
